# Day 090 Project — Indicator Dashboard

Build a technical indicator dashboard for one year of synthetic OHLCV data. Compute all five indicators with `add_indicators`, print summary statistics, and generate simple RSI-based signals. Swap `_synthetic` for `fetch_ohlcv` with `fetch_fn=None` to use real yfinance data.

In [ ]:
import pandas as pd, math

def _synthetic(n=50):
    prices = [100.0 * (1 + 0.3 * math.sin(i * 2 * math.pi / n)) for i in range(n)]
    dates  = pd.date_range("2023-01-01", periods=n, freq="B")
    close  = pd.Series(prices, index=dates)
    return pd.DataFrame({
        "Open":   close.shift(1).fillna(close.iloc[0]),
        "High":   close * 1.01,
        "Low":    close * 0.99,
        "Close":  close,
        "Volume": pd.Series([1_000_000 + i * 1_000 for i in range(n)], index=dates),
    })
def sma(series, window=20):
    return series.rolling(window=window).mean()
def ema(series, window=20):
    return series.ewm(span=window, adjust=False).mean()
def rsi(series, window=14):
    delta = series.diff()
    gain  = delta.clip(lower=0).rolling(window=window).mean()
    loss  = (-delta.clip(upper=0)).rolling(window=window).mean()
    rs    = gain / loss
    return 100 - (100 / (1 + rs))
def macd(series, fast=12, slow=26, signal=9):
    fast_ema    = ema(series, fast)
    slow_ema    = ema(series, slow)
    macd_line   = fast_ema - slow_ema
    signal_line = ema(macd_line, signal)
    return pd.DataFrame({"macd": macd_line, "signal": signal_line,
                          "histogram": macd_line - signal_line})
def bollinger_bands(series, window=20, num_std=2.0):
    middle = sma(series, window)
    std    = series.rolling(window=window).std()
    return pd.DataFrame({"upper": middle + num_std * std,
                          "middle": middle,
                          "lower": middle - num_std * std})
def add_indicators(df, sma_w=20, ema_w=20, rsi_w=14,
                   macd_fast=12, macd_slow=26, macd_sig=9,
                   bb_w=20, bb_std=2.0):
    df    = df.copy()
    close = df["Close"]
    df[f"sma_{sma_w}"] = sma(close, sma_w)
    df[f"ema_{ema_w}"] = ema(close, ema_w)
    df[f"rsi_{rsi_w}"] = rsi(close, rsi_w)
    m = macd(close, macd_fast, macd_slow, macd_sig)
    df["macd"]        = m["macd"]
    df["macd_signal"] = m["signal"]
    df["macd_hist"]   = m["histogram"]
    bb = bollinger_bands(close, bb_w, bb_std)
    df["bb_upper"]  = bb["upper"]
    df["bb_middle"] = bb["middle"]
    df["bb_lower"]  = bb["lower"]
    return df


## Step 1 — Load price data

In [ ]:
# Build 252-row synthetic OHLCV (gate-safe; swap for real yfinance data)
df = _synthetic(n=252)
print(f"Raw OHLCV: {df.shape} — {df.index[0].date()} to {df.index[-1].date()}")


## Step 2 — Enrich with all indicators

In [ ]:
enriched = add_indicators(df)
print(f"Enriched:  {enriched.shape}")
print(f"Columns:   {list(enriched.columns)}")


## Step 3 — Print summary statistics

In [ ]:
# Indicator summary for the last 100 rows (past ~5 months)
tail = enriched.tail(100)
print("\n── RSI (14) stats ──────────────────────────────")
print(tail["rsi_14"].describe().round(2))
print("\n── MACD stats ──────────────────────────────────")
print(tail[["macd", "macd_signal", "macd_hist"]].describe().round(4))
print("\n── Bollinger Band width (last 10 rows) ─────────")
width = ((enriched["bb_upper"] - enriched["bb_lower"]) / enriched["bb_middle"]).tail(10)
print(width.round(4))


## Step 4 — Generate RSI signals

In [ ]:
# Count simple RSI signals in last 200 rows
recent = enriched.tail(200).copy()
recent["rsi_signal"] = 0
recent.loc[recent["rsi_14"] < 30, "rsi_signal"] = 1   # oversold: buy
recent.loc[recent["rsi_14"] > 70, "rsi_signal"] = -1  # overbought: sell

n_buy  = (recent["rsi_signal"] == 1).sum()
n_sell = (recent["rsi_signal"] == -1).sum()
print(f"\nRSI signals (last 200 rows): {n_buy} buys, {n_sell} sells")
